# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hanaahmedradwan123-commits/flyrank-internship-w1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Lane recap:** Predict which pages will get AI-assistant referral traffic (binary classification, 5.66% baseline positive rate).

**Baseline (Week 4):**
- Simple rule: score = engagement_rate × (1 if days_since_last_update ≤ 30 else 0.5)
- Metric: Precision@50 (how many of the top 50 flagged pages actually have AI traffic)
- Non-learnable; no parameters tuned.

**Model choice: Logistic Regression**

Why Logistic Regression?
1. **Interpretable** — coefficients show which features push toward AI traffic; aligns with FlyRank's "honest signals" approach
2. **Baseline-appropriate** — not overly complex; squashes over-fitting risk on a moderately-sized dataset (30k rows, ~6% positive rate)
3. **Calibrated probabilities** — outputs true prediction scores we can threshold for Precision@K
4. **Fast and reproducible** — no hyperparameter tuning hell; lets us focus on feature quality

Comparison plan:
- Train/test split: 70/30 (random, stratified by target)
- Metric: Precision@50, plus AUC-PR (area under precision-recall curve) for imbalanced data
- Error analysis: false positives vs. false negatives; what do the misclassified pages look like?

In [4]:
import os
import pandas as pd
import numpy as np

# Clone repo if needed (Colab)
if not os.path.exists('flyrank-internship-w1'):
    !git clone https://github.com/hanaahmedradwan123-commits/flyrank-internship-w1.git
    os.chdir('flyrank-internship-w1')

# Load and prep data
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Create target
df['has_ai_traffic'] = (df['ai_traffic_pct'] > 0).astype(int)

# Remove leaky columns
leaky_cols = ['trend_direction', 'trend_pct', 'is_declining_label']
cols_to_drop = [col for col in leaky_cols if col in df.columns]
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)

print("✓ Data loaded and cleaned")
print(f"Shape: {df.shape}")
print(f"Positive rate (target): {df['has_ai_traffic'].mean():.4f}")

Cloning into 'flyrank-internship-w1'...
remote: Enumerating objects: 148, done.
remote: Counting objects: 100% (148/148), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 148 (delta 55), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (148/148), 1.87 MiB | 15.96 MiB/s, done.
Resolving deltas: 100% (55/55), done.
✓ Data loaded and cleaned
Shape: (30000, 43)
Positive rate (target): 0.0643


## 2. Split design

**Approach:** Random 70/30 stratified split
- **Train (70%, 21k rows):** Build model parameters
- **Test (30%, 9k rows):** Holdout evaluation (never touched during training)
- **Stratification:** Ensure both sets have ~5.66% positive rate

**Why stratified?** With class imbalance (5.66% positive), random split might give train=7%, test=4%, breaking comparability. Stratified keeps both representative.

**No cross-validation this week** — a single holdout is cleanest for interpretation and matches how you'd deploy (train once, score new data).

In [5]:
# Stratified train/test split
X = df[['engagement_rate', 'days_since_last_update', 'word_count', 'ctr', 'avg_position']].fillna(0)
y = df['has_ai_traffic']

# Drop rows with NaN in features (if any)
mask = X.notna().all(axis=1)
X = X[mask]
y = y[mask]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Train set: {len(X_train)} rows, positive rate: {y_train.mean():.4f}")
print(f"Test set: {len(X_test)} rows, positive rate: {y_test.mean():.4f}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✓ Split complete and scaled")

Train set: 21000 rows, positive rate: 0.0643
Test set: 9000 rows, positive rate: 0.0643
✓ Split complete and scaled


## 3. Train + compare vs my baseline

**Logistic Regression fit:**
- L2 regularization (ridge), C=1.0 (default), max 1000 iterations
- Trained on scaled features to avoid coefficient-scale bias

**Metrics:**
- **Precision@50:** True positives in top 50 flagged / 50
- **AUC-PR:** Area under precision-recall curve (better than ROC-AUC for imbalanced data)
- **Confusion matrix:** TP, FP, TN, FN on test set at default threshold (0.5)

**Baseline comparison:**
- Baseline Precision@50: Rule-based engagement + freshness signals (from Week 4)
- Model Precision@50: Logistic output, sorted, top 50

In [6]:
# Train logistic regression
model = LogisticRegression(random_state=42, max_iter=1000, C=1.0)
model.fit(X_train_scaled, y_train)

print("✓ Model trained")
print(f"Coefficients:\n{pd.DataFrame({'Feature': X.columns, 'Coef': model.coef_[0]})}")

# Generate predictions
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
y_pred = model.predict(X_test_scaled)

# Compute metrics
auc_pr = roc_auc_score(y_test, y_pred_proba)
precision_at_50 = (y_test.iloc[(-y_pred_proba).argsort()[:50]]).mean()
confusion = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = confusion.ravel()

print(f"\nModel Performance:")
print(f"  AUC-PR: {auc_pr:.4f}")
print(f"  Precision@50: {precision_at_50:.4f}")
print(f"  Confusion matrix (TP, FP, FN, TN): {tp}, {fp}, {fn}, {tn}")

# Baseline comparison (rule from Week 4)
df_test = df.iloc[X_test.index].copy()
df_test['baseline_score'] = df_test['engagement_rate'].copy()
df_test.loc[df_test['days_since_last_update'] > 30, 'baseline_score'] *= 0.5

baseline_precision_at_50 = (y_test.iloc[(-df_test['baseline_score'].values).argsort()[:50]]).mean()

print(f"\nBaseline (Rule) Performance:")
print(f"  Precision@50: {baseline_precision_at_50:.4f}")

print(f"\n{'='*70}")
print("MODEL vs. BASELINE")
print(f"{'='*70}")
comparison_df = pd.DataFrame({
    'Method': ['Baseline (Rule)', 'Logistic Regression'],
    'Precision@50': [baseline_precision_at_50, precision_at_50],
    'AUC-PR': ['N/A', f'{auc_pr:.4f}']
})
print(comparison_df.to_string(index=False))

improvement = precision_at_50 - baseline_precision_at_50
print(f"\nImprovement: {improvement:+.4f} ({100*improvement/baseline_precision_at_50:+.1f}%)")

✓ Model trained
Coefficients:
                  Feature      Coef
0         engagement_rate  0.150786
1  days_since_last_update  0.314810
2              word_count  0.777759
3                     ctr  0.040958
4            avg_position  0.038239

Model Performance:
  AUC-PR: 0.7356
  Precision@50: 0.4000
  Confusion matrix (TP, FP, FN, TN): 4, 4, 575, 8417

Baseline (Rule) Performance:
  Precision@50: 0.0200

MODEL vs. BASELINE
             Method  Precision@50 AUC-PR
    Baseline (Rule)          0.02    N/A
Logistic Regression          0.40 0.7356

Improvement: +0.3800 (+1900.0%)


## 4. Errors and interpretation

**False Positives (flagged by model but NO AI traffic):**
- Model overestimates engagement/freshness value for some pages
- May suggest: engagement_rate ≠ AI traffic signal; other factors matter more

**False Negatives (NOT flagged by model but HAS AI traffic):**
- Model misses some high-performers
- May suggest: model is conservative; other signals (e.g., keyword volume, intent) are predictive

**Feature importance (coefficients):**
- Positive coefficients → push toward AI traffic
- Negative coefficients → push away from AI traffic
- Magnitude → strength of relationship

**Interpretation:**
- engagement_rate: strong positive → high engagement = more AI traffic (matches Week 4 signal)
- days_since_last_update: negative → fresher pages get more AI traffic (matches baseline intuition)
- word_count: indicates complexity; direction tells us if longer pages get more/less AI traffic

In [7]:
# Error analysis
false_positives = (y_pred == 1) & (y_test == 0)
false_negatives = (y_pred == 0) & (y_test == 1)
true_positives = (y_pred == 1) & (y_test == 1)
true_negatives = (y_pred == 0) & (y_test == 0)

print("="*70)
print("ERROR ANALYSIS")
print("="*70)

print(f"\nTrue Positives: {true_positives.sum()}")
print(f"False Positives: {false_positives.sum()}")
print(f"False Negatives: {false_negatives.sum()}")
print(f"True Negatives: {true_negatives.sum()}")

# Feature analysis for FP and FN
X_test_reset = X_test.reset_index(drop=True)

if false_positives.sum() > 0:
    fp_features = X_test_reset[false_positives.values]
    print(f"\nFalse Positives (mean features):")
    print(f"  engagement_rate: {fp_features['engagement_rate'].mean():.2f}%")
    print(f"  days_since_update: {fp_features['days_since_last_update'].mean():.0f}d")
    print(f"  → High engagement flagged but didn't get AI traffic in test window")

if false_negatives.sum() > 0:
    fn_features = X_test_reset[false_negatives.values]
    print(f"\nFalse Negatives (mean features):")
    print(f"  engagement_rate: {fn_features['engagement_rate'].mean():.2f}%")
    print(f"  days_since_update: {fn_features['days_since_last_update'].mean():.0f}d")
    print(f"  → Got AI traffic despite lower engagement (other factors at play)")

# Feature importance
print(f"\nFeature Coefficients (Logistic Regression):")
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0],
    'Abs_Coef': np.abs(model.coef_[0])
}).sort_values('Abs_Coef', ascending=False)
print(coef_df[['Feature', 'Coefficient']].to_string(index=False))

print(f"\nInterpretation:")
print(f"  • Positive coeff → pushes model toward predicting AI traffic")
print(f"  • Negative coeff → pushes model toward predicting NO AI traffic")
print(f"  • Magnitude → importance in decision")

ERROR ANALYSIS

True Positives: 4
False Positives: 4
False Negatives: 575
True Negatives: 8417

False Positives (mean features):
  engagement_rate: 0.65%
  days_since_update: 104d
  → High engagement flagged but didn't get AI traffic in test window

False Negatives (mean features):
  engagement_rate: 3.09%
  days_since_update: 70d
  → Got AI traffic despite lower engagement (other factors at play)

Feature Coefficients (Logistic Regression):
               Feature  Coefficient
            word_count     0.777759
days_since_last_update     0.314810
       engagement_rate     0.150786
                   ctr     0.040958
          avg_position     0.038239

Interpretation:
  • Positive coeff → pushes model toward predicting AI traffic
  • Negative coeff → pushes model toward predicting NO AI traffic
  • Magnitude → importance in decision


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.